<a href="https://colab.research.google.com/github/DavidOprea/LiftScope/blob/machine-learning-model/Gym_Classification_Model_LiftScope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -Uqq fastbook
import fastbook
from fastbook import *
!pip install fastprogress==1.0.3
!pip install fastai==2.7.13

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Copy your two hand-curated folders from Drive:
#   Gym_Machines_Better_Train  -> becomes train/
#   Gym_Machines_Better_Eval   -> becomes valid/  (will be converted from HEIC below)
!cp -r "/content/drive/MyDrive/Gym_Machines_Better_Train" "/content/"
!cp -r "/content/drive/MyDrive/Gym_Machines_Better_Eval"  "/content/"

!ls -la "/content/"

In [ ]:
!pip install pillow_heif Pillow

In [ ]:
from PIL import Image
import pillow_heif
import os

pillow_heif.register_heif_opener()

'''
IPhone images are originall in the HEIC format and our model
doesn't use that format, so we convert them to JPEG.
'''

def convert_heic_to_jpg(input_file, output_file):
    try:
        image = Image.open(input_file)
        if image.mode != "RGB":
            image = image.convert("RGB")
        image.save(output_file, format="JPEG")
    except Exception as e:
        print(f"Error converting {input_file}: {e}")

base_path = "/content/Gym_Machines_Better_Eval"
add_path  = "/content/Gym_Machines_Better_Eval_Good"

for cls in os.listdir(base_path):
    os.makedirs(os.path.join(add_path, cls), exist_ok=True)

for cls in os.listdir(base_path):
    folder = os.path.join(base_path, cls)
    for img in os.listdir(folder):
        src  = os.path.join(folder, img)
        dst  = os.path.join(add_path, cls, img.split(".")[0] + ".jpg")
        convert_heic_to_jpg(src, dst)

print("Done converting eval images.")

In [ ]:
import shutil
from pathlib import Path

# Create the training and validation folders

train_src = Path("/content/Gym_Machines_Better_Train")
valid_src  = Path("/content/Gym_Machines_Better_Eval_Good")
dest       = Path("/content/Gym_Machines_Data")

for class_folder in train_src.iterdir():
    if not class_folder.is_dir():
        continue
    out = dest / "train" / class_folder.name
    out.mkdir(parents=True, exist_ok=True)
    for img in class_folder.iterdir():
        shutil.copy(str(img), str(out / img.name))

for class_folder in valid_src.iterdir():
    if not class_folder.is_dir():
        continue
    out = dest / "valid" / class_folder.name
    out.mkdir(parents=True, exist_ok=True)
    for img in class_folder.iterdir():
        shutil.copy(str(img), str(out / img.name))

# Verify the layout looks right
print("Sample structure:")
for p in sorted(dest.glob("*/*"))[:12]:
    count = len(list(p.iterdir())) if p.is_dir() else "(file)"
    print(f"  {p}  ->  {count} images")

In [ ]:
# Build out the data preprocessing of our images

gym_equipments_block = DataBlock(
    blocks=(ImageBlock(cls=PILImage), CategoryBlock),
    get_items=get_image_files,
    splitter=GrandparentSplitter(train_name='train', valid_name='valid'),
    get_y=parent_label,
    item_tfms=Resize(460, "squish"),
    batch_tfms=[*aug_transforms(size=224, mult=1.0, min_scale=0.75)]
)

path = "/content/Gym_Machines_Data"
dls  = gym_equipments_block.dataloaders(path, num_workers=2)

print(f"Classes ({dls.c}): {dls.vocab}")

In [ ]:
dls.show_batch(max_n=4)

In [ ]:
# Create and train the model

!pip install timm

import timm
learn = vision_learner(dls, 'mobilenetv3_large_100', metrics=accuracy)

learn.fine_tune(4, freeze_epochs=2, base_lr=0.001)

In [ ]:
preds, targs = learn.tta()
print("TTA Accuracy:", accuracy(preds, targs))

In [ ]:
# .pkl exports, in case you want to use the model in a different Python file.

learn.export('purdue_gym_model.pkl')

from google.colab import files
files.download('purdue_gym_model.pkl')

In [ ]:
# Libraries used to export a tflite for the model

!pip install onnx onnxscript onnxruntime onnx2tf tensorflow

In [ ]:
import torch
import torch.nn as nn

# Exporting the ONNX file and changing the how the model
# is exported so React Native understands how it works.

class ONNXSafeConcatPool(nn.Module):
    def forward(self, x):
        # keepdim=False collapses H and W immediately -> [B, C]
        ap = torch.mean(x, dim=[-2, -1], keepdim=False)
        mp = torch.amax(x, dim=[-2, -1], keepdim=False)
        return torch.cat([mp, ap], dim=1)  # -> [B, 2C]

class MobileReadyModel(nn.Module):
    def __init__(self, core_model):
        super().__init__()
        self.core_model = core_model
        # Patch FastAI pooling layer
        self.core_model[1][0] = ONNXSafeConcatPool()
        # Patch FastAI dynamic Flatten with a static one
        self.core_model[1][1] = nn.Flatten(start_dim=1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.core_model(x)
        return self.softmax(x)

base_model   = learn.model.eval()
mobile_model = MobileReadyModel(base_model).cpu()
dummy_input  = torch.randn(1, 3, 224, 224).cpu()

torch.onnx.export(
    mobile_model,
    dummy_input,
    "gym_model_mobile.onnx",
    export_params=True,
    opset_version=14,
    input_names=['input'],
    output_names=['output'],
)
print("ONNX export done.")

In [ ]:
!onnx2tf -i gym_model_mobile.onnx -o gym_model_tflite -b 1
print("TFLite conversion done.")